#Libraries  

In [1]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
print(nltk.__version__)


3.10.3


REQUIRED LIBRARIES

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

DATA GATHERING

In [4]:
df=pd.read_csv("WELFake_Dataset.csv")
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


DATA ANALYSIS

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  72134 non-null  int64
 1   title       71576 non-null  str  
 2   text        72095 non-null  str  
 3   label       72134 non-null  int64
dtypes: int64(2), str(2)
memory usage: 2.2 MB


In [6]:
df['label'].value_counts()

label
1    37106
0    35028
Name: count, dtype: int64

In [7]:
df.shape

(72134, 4)

In [8]:
df.isna().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [9]:
df=df.dropna()

In [10]:
df.isna().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [11]:
df.shape

(71537, 4)

In [12]:
#renaming bullshit
df.reset_index(drop=True, inplace=True)
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
2,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
3,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
4,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1


In [13]:
df['title'][0]

'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'

In [14]:
df.rename(columns={'Unnamed: 0': 'index'}, inplace=True)
df.head()

,index,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
2,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
3,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
4,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1


In [15]:
df.drop(columns=['index'], inplace=True)

In [16]:
df.insert(0, 'index', range(len(df)))
df.head(10)

,index,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
2,2,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
3,3,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
4,4,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1
5,5,DR BEN CARSON TARGETED BY THE IRS: “I never ha...,DR. BEN CARSON TELLS THE STORY OF WHAT HAPPENE...,1
6,6,HOUSE INTEL CHAIR On Trump-Russia Fake Story: ...,,1
7,7,Sports Bar Owner Bans NFL Games…Will Show Only...,"The owner of the Ringling Bar, located south o...",1
8,8,Latest Pipeline Leak Underscores Dangers Of Da...,"FILE – In this Sept. 15, 2005 file photo, the ...",1
9,9,GOP Senator Just Smacked Down The Most Puncha...,The most punchable Alt-Right Nazi on the inter...,1


In [17]:
df=df.drop(['text'],axis=1)
df.head()

,index,title,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
2,2,"Bobby Jindal, raised Hindu, uses story of Chri...",0
3,3,SATAN 2: Russia unvelis an image of its terrif...,1
4,4,About Time! Christian Group Sues Amazon and SP...,1


DATA PREPROCESSING

In [18]:
# Ensure you have the required datasets downloaded
nltk.download('stopwords') 
nltk.download('wordnet')               # 2. Download the WordNet dataset

# Initialize the lemmatizer and stop words
lm = WordNetLemmatizer()               # 3. Define 'lm' here to fix the NameError
stop_words = set(stopwords.words('english'))

corpus = []
for i in range(len(df)):
    review = re.sub('[^a-zA-Z0-9]', ' ', df['title'][i])
    review = review.lower()
    review = review.split()
    review = [lm.lemmatize(x) for x in review if x not in stop_words]
    review = " ".join(review)
    corpus.append(review)

[nltk_data] Downloading package stopwords to C:\Users\Anurag
[nltk_data]     Tripathi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Anurag
[nltk_data]     Tripathi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [19]:
len(corpus)

71537

In [20]:
df['title'][0]

'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'

In [21]:
corpus[0]

'law enforcement high alert following threat cop white 9 11by blacklivesmatter fyf911 terrorist video'

VECTORIZATION         convert text data into vector

In [22]:
tf = TfidfVectorizer(max_features=5000, dtype=np.float32)
x = tf.fit_transform(corpus).toarray()
x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(71537, 5000), dtype=float32)

In [23]:
import psutil

# Get system memory statistics
memory = psutil.virtual_memory()

# Convert bytes to Gigabytes (GB) for easy reading
total_gb = memory.total / (1024**3)
available_gb = memory.available / (1024**3)
used_gb = memory.used / (1024**3)

print(f"Total System RAM:  {total_gb:.2f} GB")
print(f"Used System RAM:   {used_gb:.2f} GB ({memory.percent}%)")
print(f"Available/Free RAM: {available_gb:.2f} GB")

Total System RAM:  15.65 GB
Used System RAM:   8.34 GB (53.3%)
Available/Free RAM: 7.30 GB


In [24]:
import sys

# Get all global variables and filter out system built-ins
local_vars = {k: v for k, v in globals().items() if not k.startswith('_')}

print(f"{'Variable Name':<25} | {'Memory Usage':<15}")
print("-" * 45)

for var_name, var_value in sorted(local_vars.items(), key=lambda x: sys.getsizeof(x[1]), reverse=True):
    # Get size in bytes and convert to Megabytes
    size_mb = sys.getsizeof(var_value) / (1024 ** 2)
    
    # Only print variables taking up more than 0.1 MB to keep it clean
    if size_mb > 0.1:
        print(f"{var_name:<25} | {size_mb:.2f} MB")


Variable Name             | Memory Usage   
---------------------------------------------
x                         | 1364.46 MB
df                        | 12.19 MB
corpus                    | 0.60 MB


In [25]:
import gc

# Delete the massive array
del x

# Force Python's garbage collector to empty the cache immediately
gc.collect()


20

In [26]:
import sys

# Get all global variables and filter out system built-ins
local_vars = {k: v for k, v in globals().items() if not k.startswith('_')}

print(f"{'Variable Name':<25} | {'Memory Usage':<15}")
print("-" * 45)

for var_name, var_value in sorted(local_vars.items(), key=lambda x: sys.getsizeof(x[1]), reverse=True):
    # Get size in bytes and convert to Megabytes
    size_mb = sys.getsizeof(var_value) / (1024 ** 2)
    
    # Only print variables taking up more than 0.1 MB to keep it clean
    if size_mb > 0.1:
        print(f"{var_name:<25} | {size_mb:.2f} MB")

Variable Name             | Memory Usage   
---------------------------------------------
df                        | 12.19 MB
corpus                    | 0.60 MB


In [27]:
import psutil

# Get system memory statistics
memory = psutil.virtual_memory()

# Convert bytes to Gigabytes (GB) for easy reading
total_gb = memory.total / (1024**3)
available_gb = memory.available / (1024**3)
used_gb = memory.used / (1024**3)

print(f"Total System RAM:  {total_gb:.2f} GB")
print(f"Used System RAM:   {used_gb:.2f} GB ({memory.percent}%)")
print(f"Available/Free RAM: {available_gb:.2f} GB")

Total System RAM:  15.65 GB
Used System RAM:   8.35 GB (53.4%)
Available/Free RAM: 7.30 GB


In [28]:
tf = TfidfVectorizer(max_features=5000, dtype=np.float32)
x = tf.fit_transform(corpus).toarray()
x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(71537, 5000), dtype=float32)

In [29]:
y=df['label']
y.head()

0    1
1    1
2    0
3    1
4    1
Name: label, dtype: int64

DATA SPLITING INTO TRAIN AND TEST

In [30]:
x_train, x_test, y_train, y_test= train_test_split(x,y, test_size = 0.23, random_state = 10, stratify = y)

In [31]:
len(x_train), len(y_train)

(55083, 55083)

In [32]:
len(x_test), len(y_test)

(16454, 16454)

MODEL BUILDING

In [33]:
rf= RandomForestClassifier()
rf.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

MODEL EVALUATION

In [34]:
from sklearn.metrics import accuracy_score
y_pred=rf.predict(x_test)
acc_scor = accuracy_score(y_test, y_pred)
acc_scor 

0.8972894129087152

In [50]:
class Evaluation:
    
    def __init__(self, model, x_train, x_test, y_train, y_test):
        self.model=model
        self.x_train= x_train
        self.x_test= x_test
        self.y_train= y_train
        self.y_test= y_test

    def train_evaluation(self):
        y_pred_train = self.model.predict(self.x_train)
        
        acc_scr_train = accuracy_score(self.y_train, y_pred_train)
        print("Accuracy Score on Training Dataset: ", acc_scr_train)
              
        con_mat_train = confusion_matrix(self.y_train, y_pred_train)
        print("Confusion Matrix on Training Dataset: ", con_mat_train)
              
        class_rep_train = classification_report(self.y_train, y_pred_train)
        print("Classification Report on Training Dataset: ", class_rep_train)
        
    def test_evaluation(self):
        y_pred_test = self.model.predict(self.x_test)
        
        acc_scr_test = accuracy_score(self.y_test, y_pred_test)
        print("Accuracy Score on Testing Dataset: ", acc_scr_test)
              
        con_mat_test = confusion_matrix(self.y_test, y_pred_test)
        print("Confusion Matrix on Testing Dataset: ", con_mat_test)
              
        class_rep_test = classification_report(self.y_test, y_pred_test)
        print("Classification Report on Testing Dataset: ", class_rep_test)

In [46]:
#Checking the Accuracy on the training Dataset

Evaluation(rf, x_train, x_test, y_train, y_test).train_evaluation()

Accuracy Score on Training Dataset:  0.999745838098869
Confusion Matrix on Training Dataset:  [[26962     9]
 [    5 28107]]
Classification Report on Training Dataset:                precision    recall  f1-score   support

           0       1.00      1.00      1.00     26971
           1       1.00      1.00      1.00     28112

    accuracy                           1.00     55083
   macro avg       1.00      1.00      1.00     55083
weighted avg       1.00      1.00      1.00     55083



In [51]:
#Checking the Accuracy on the testing Dataset

Evaluation(rf, x_train, x_test, y_train, y_test).test_evaluation()

Accuracy Score on Testing Dataset:  0.8972894129087152
Confusion Matrix on Testing Dataset:  [[7160  897]
 [ 793 7604]]
Classification Report on Testing Dataset:                precision    recall  f1-score   support

           0       0.90      0.89      0.89      8057
           1       0.89      0.91      0.90      8397

    accuracy                           0.90     16454
   macro avg       0.90      0.90      0.90     16454
weighted avg       0.90      0.90      0.90     16454



PREDICTION PIPELINE

In [65]:
class Preprocessing:
    def __init__(self, data):
        self.data = data
        self.lm = WordNetLemmatizer()
        self.stopwords_list = stopwords.words('english')

    def text_preprocessing_user(self):
        preprocess_data = []
        # Use self.data here
        for item in self.data:
            review = re.sub('[^a-zA-Z0-9]', ' ', item)
            review = review.lower()
            review = review.split()
            review = [self.lm.lemmatize(x) for x in review if x not in self.stopwords_list]
            review = " ".join(review)
            preprocess_data.append(review)
        return preprocess_data 

In [53]:
df['title'][1]

'UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]'

In [66]:
data = 'UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]'
Preprocessing(data).text_preprocessing_user()

['u',
 'n',
 'b',
 'e',
 'l',
 '',
 'e',
 'v',
 '',
 'b',
 'l',
 'e',
 '',
 '',
 '',
 'b',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'r',
 'n',
 'e',
 '',
 '',
 'g',
 'e',
 'n',
 'e',
 'r',
 '',
 'l',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'c',
 'h',
 '',
 'r',
 'l',
 '',
 '',
 '',
 'e',
 '',
 'r',
 '',
 '',
 '',
 'e',
 'r',
 '',
 '',
 'w',
 'e',
 'r',
 'e',
 '',
 '',
 'p',
 'e',
 '',
 'c',
 'e',
 'f',
 'u',
 'l',
 '',
 '',
 'p',
 'r',
 '',
 '',
 'e',
 '',
 '',
 'e',
 'r',
 '',
 '',
 '',
 'n',
 '',
 'h',
 'e',
 'r',
 '',
 'h',
 '',
 '',
 'e',
 '',
 '',
 '',
 '',
 '',
 'e',
 '',
 '',
 'f',
 '',
 'n',
 '',
 'r',
 '',
 'h',
 '',
 'c',
 '',
 'r',
 '',
 'l',
 '',
 'n',
 '',
 '',
 '',
 'v',
 '',
 '',
 'e',
 '',
 '']

In [61]:
class Prediction:

    def __init__(self, pred_data, model):
        self.pred_data = pred_data
        self.model = model

    def prediction_model(self):
        preprocess_data = Preprocessing(self.pred_data).text_preprocessing_user()
        data = tf.transform(preprocess_data)
        prediction = self.model.predict(data)

        if prediction [0] ==0 : 
            return "THE NEWS IS FAKE"
        else:
            return"THE NEWS IS REAL"

In [62]:
df['title'][2026]

'Police Turn In Badges Rather Than Incite Violence Against Standing Rock Protestors'

In [67]:
data= 'Police Turn In Badges Rather Than Incite Violence Against Standing Rock Protestors'
Prediction(data,rf).prediction_model()

'THE NEWS IS REAL'